In [ ]:
!pip install boto3

In [1]:
import json
import csv
import pandas as pd
import matplotlib.pyplot as plt
import boto3
from io import BytesIO
import numpy as np
from urllib.parse import urlparse
import math
import os

%matplotlib inline

Matplotlib is building the font cache; this may take a moment.


In [2]:
client = boto3.client("s3")

def list_all_objects_in_s3(bucketName, prefixName=None):
    """Get a list of objects in an S3 bucket path.
    @params:
    bucketName : s3 bucket
    prefixName : s3 prefix to bucket
    returns : list of objects in s3 path
    [example of S3 bucket with prefix f"s3://{bucketName}/{prefix}"]
    """
    paginator = client.get_paginator('list_objects_v2')
    if prefixName:
        pages = paginator.paginate(Bucket=bucketName, StartAfter=prefixName)
    else:
        pages = paginator.paginate(Bucket=bucketName)
    list_of_files=[]
    for page in pages:
        for obj in page['Contents']:
            files = bucketName+'/'+obj['Key']
            if prefixName:
                if bucketName+'/'+prefixName in files:
                    if bucketName+'/'+prefixName+'/' !=files:
                        list_of_files.append(files)
            else:
                list_of_files.append(files)
    return list_of_files

def read_json_file_from_s3(bucketName, prefixName, fileName):
    """ returns json file content
        @param
        bucketName : s3 bucket
        prefixName : s3 prefix to bucket
        fileName   : input json file Name
        [example of S3 bucket with prefix f"s3://{bucketName}/{prefix}"]
    """
    result = client.list_objects(Bucket = bucketName, Prefix=prefixName)
    for o in result.get('Contents'):
        data = client.get_object(Bucket=bucketName, Key=o.get('Key'))
        contents = data['Body'].read()
    return json.loads(contents.decode("utf-8"))

def coco_to_yolo(x1, y1, w, h, image_w, image_h):
    return [(x1 + w/2)/image_w, (y1 + h/2)/image_h, w/image_w, h/image_h]

def s3_copy(srcbucket, srckey, destbucket, destkey):
    try:
        s3 = boto3.resource('s3')
        copy_source = {
            'Bucket': srcbucket,
            'Key': srckey
        }
        s3.meta.client.copy(copy_source, destbucket, destkey)
    except Exception as e:
        print('s3 copy failed. '+ str(e)+'for file: '+copy_source)
    return 

def to_s3_npy(data: np.array, s3_uri: str):
    # s3_uri looks like f"s3://{BUCKET_NAME}/{KEY}"
    """
    @params 
    data : file content 
    s3_uri : s3 file path along with name [example of S3 bucket with prefix f"s3://{bucketName}/{prefix}/{fileName.npy}"]
    """
    try:
        bytes_ = BytesIO()
        np.save(bytes_, data, allow_pickle=True)
        bytes_.seek(0)
        parsed_s3 = urlparse(s3_uri)
        client.upload_fileobj(
            Fileobj=bytes_, Bucket=parsed_s3.netloc, Key=parsed_s3.path[1:]
        )
    except Exception as e: # work on python 3.x
        print('Failed to write numpy array to S3: '+ str(e))
    return True

def from_s3_npy(s3_uri: str):
    """
    @params 
    data : file content 
    s3_uri : s3 file path along with name [example of S3 bucket with prefix f"s3://{bucketName}/{prefix}/{fileName.npy}"]
    """
    try:
        bytes_ = BytesIO()
        parsed_s3 = urlparse(s3_uri)
        client.download_fileobj(
            Fileobj=bytes_, Bucket=parsed_s3.netloc, Key=parsed_s3.path[1:]
        )
        bytes_.seek(0)
        np_data=np.load(bytes_, allow_pickle=True)
    
    except Exception as e: # work on python 3.x
        print('Failed to read numpy array from S3: '+ str(e))
    return np_data

In [3]:
list_all_objects_in_s3("skm-dataset", "skm-tea/qdess/v1-release/annotations/v1.0.0")

['skm-dataset/skm-tea/qdess/v1-release/annotations/v1.0.0/test.json',
 'skm-dataset/skm-tea/qdess/v1-release/annotations/v1.0.0/train.json',
 'skm-dataset/skm-tea/qdess/v1-release/annotations/v1.0.0/val.json']

In [4]:
filenames = ["test.json", "train.json", "val.json"]
print("annotation files:", filenames)

annotation files: ['test.json', 'train.json', 'val.json']


In [5]:
train = pd.read_csv("train_annotations.csv")
val = pd.read_csv("val_annotations.csv")
test = pd.read_csv("test_annotations.csv")

In [6]:
test["scan_id"].unique()

array(['MTR_173', 'MTR_176', 'MTR_178', 'MTR_188', 'MTR_196', 'MTR_198',
       'MTR_219', 'MTR_221', 'MTR_218', 'MTR_199', 'MTR_224', 'MTR_227',
       'MTR_235', 'MTR_237', 'MTR_240', 'MTR_241', 'MTR_244', 'MTR_248',
       'MTR_005', 'MTR_006', 'MTR_030', 'MTR_034', 'MTR_048', 'MTR_052',
       'MTR_065', 'MTR_066', 'MTR_079', 'MTR_080', 'MTR_096', 'MTR_099',
       'MTR_120', 'MTR_126', 'MTR_144', 'MTR_156', 'MTR_158'],
      dtype=object)

In [7]:
val["scan_id"].unique()

array(['MTR_133', 'MTR_149', 'MTR_220', 'MTR_037', 'MTR_083', 'MTR_110',
       'MTR_139', 'MTR_182', 'MTR_208', 'MTR_203', 'MTR_047', 'MTR_069',
       'MTR_071', 'MTR_112', 'MTR_154', 'MTR_184', 'MTR_207', 'MTR_222',
       'MTR_023', 'MTR_044', 'MTR_062', 'MTR_115', 'MTR_130', 'MTR_152',
       'MTR_167', 'MTR_026', 'MTR_045'], dtype=object)

In [8]:
train["scan_id"].unique()

array(['MTR_104', 'MTR_105', 'MTR_179', 'MTR_194', 'MTR_046', 'MTR_174',
       'MTR_177', 'MTR_211', 'MTR_212', 'MTR_225', 'MTR_015', 'MTR_159',
       'MTR_018', 'MTR_172', 'MTR_145', 'MTR_057', 'MTR_189', 'MTR_001',
       'MTR_008', 'MTR_016', 'MTR_020', 'MTR_033', 'MTR_040', 'MTR_087',
       'MTR_095', 'MTR_097', 'MTR_113', 'MTR_164', 'MTR_129', 'MTR_193',
       'MTR_210', 'MTR_243', 'MTR_051', 'MTR_053', 'MTR_077', 'MTR_098',
       'MTR_140', 'MTR_142', 'MTR_150', 'MTR_163', 'MTR_183', 'MTR_190',
       'MTR_200', 'MTR_201', 'MTR_204', 'MTR_209', 'MTR_229', 'MTR_232',
       'MTR_251', 'MTR_056', 'MTR_094', 'MTR_100', 'MTR_107', 'MTR_124',
       'MTR_135', 'MTR_161', 'MTR_175', 'MTR_186', 'MTR_230', 'MTR_233',
       'MTR_236', 'MTR_245', 'MTR_019', 'MTR_049', 'MTR_061', 'MTR_084',
       'MTR_093', 'MTR_101'], dtype=object)

In [9]:
test["image"] = test["scan_id"].str.cat(test["sl"].values.astype(str), sep="_")


In [10]:
len(test.image.unique())

3110

In [11]:
val["image"] = val["scan_id"].str.cat(val["sl"].values.astype(str), sep="_")

In [12]:
len(list(val.image.unique()))

1801

In [13]:
len(list_all_objects_in_s3("skm-yolo", "valid/images"))

1801

In [14]:
train["image"] = train["scan_id"].str.cat(train["sl"].values.astype(str), sep="_")

In [15]:
len(list(train.image.unique()))

4469

In [16]:
len(list_all_objects_in_s3("skm-yolo", "train/images"))

4359

In [17]:
val["supercat_id"] = val["supercat_id"] - 1

In [18]:
def write_file_to_s3(bucketName, fileContent, fileName, fileType="dataframe", prefixName=None):
    try:
        if prefixName:
            fileName=prefixName+'/'+fileName
        if fileType=='dataframe':
            fileContent.to_csv('s3://'+bucketName+'/'+fileName+'.txt', index=False, header=None)
        else:
            response = client.put_object(Bucket=bucketName, #'testshi'
                                         Body=fileContent, # '{"content": "bytes or seekable file-like object"}',
                                         Key=fileName) #'test.json' # 'Object key for which the PUT operation was initiated'
        print('Writing to S3 completed for ',fileName)
    except Exception as e: # work on python 3.x
        print('Failed to write numpy to S3: '+ str(e))
    return

s3 = boto3.resource('s3')
def read_file_from_s3(bucket, file):
    try:
        obj = s3.Object(bucket, file)
        data = obj.get()["Body"]
        
    except Exception as e: # work on python 3.x
        print('Failed to read file from S3: '+ str(e))
    return data.read().decode(encoding='utf-8')

def set_key(dictionary, key, value):
    if type(dictionary[key]) == list:
        dictionary[key].append(value)
    else:
        dictionary[key] = [dictionary[key], value]

In [21]:
len(val.image.unique())

1801

In [20]:
val_df = val

In [ ]:
for ind in val_df.index:
    dict_row = val_df.iloc[ind].to_dict()
    dict_row["yolo_bbox"] = [float(x.strip(" []")) for x in dict_row["yolo_bbox"].split(",")]
    new_dict_row = {}
    df = pd.DataFrame()
    file_list = [file.split("/")[-1] for file in list_all_objects_in_s3("skm-yolo", "valid/labels")]
    if not np.isnan(dict_row["yolo_bbox"]).any():
        new_dict_row["class"] = dict_row["supercat_id"]
        new_dict_row["yolo_cx"] = dict_row["yolo_bbox"][0]
        new_dict_row["yolo_cy"] = dict_row["yolo_bbox"][1]
        new_dict_row["yolo_w"] = dict_row["yolo_bbox"][2]
        new_dict_row["yolo_h"] = dict_row["yolo_bbox"][3]        
        outputFileName = dict_row["image"]
        if (outputFileName+".txt") in file_list:
            content = read_file_from_s3("skm-yolo", "valid/labels/"+outputFileName+".txt").strip("\n")
            for sub_content in content.split("\n"):
                set_key(new_dict_row, "class", int(sub_content.split(",")[0]))
                set_key(new_dict_row, "yolo_cx", float(sub_content.split(",")[1]))
                set_key(new_dict_row, "yolo_cy", float(sub_content.split(",")[2]))
                set_key(new_dict_row, "yolo_w", float(sub_content.split(",")[3]))
                set_key(new_dict_row, "yolo_h", float(sub_content.split(",")[4]))
            df = pd.concat([df, pd.DataFrame(new_dict_row)], ignore_index=True)
            write_file_to_s3("skm-yolo", df, outputFileName, "dataframe", "valid/labels")
            
        else:
            df = pd.concat([df, pd.DataFrame([new_dict_row])], ignore_index=True)
            write_file_to_s3("skm-yolo", df, outputFileName, "dataframe", "valid/labels")
            
        if len(list_all_objects_in_s3("skm-yolo", "valid/labels")) % 20 == 0:
            print(len(list_all_objects_in_s3("skm-yolo", "valid/labels")))

In [23]:
len(list_all_objects_in_s3("skm-yolo", "valid/labels/"))

1801

In [24]:
train["supercat_id"] = train["supercat_id"] - 1

In [25]:
len(train.image.unique())

4469

In [26]:
train_df = train

In [27]:
for ind in train_df.index:
    dict_row = train_df.iloc[ind].to_dict()
    dict_row["yolo_bbox"] = [float(x.strip(" []")) for x in dict_row["yolo_bbox"].split(",")]
    new_dict_row = {}
    df = pd.DataFrame()
    file_list = [file.split("/")[-1] for file in list_all_objects_in_s3("skm-yolo", "train/labels")]
    if not np.isnan(dict_row["yolo_bbox"]).any():
        new_dict_row["class"] = dict_row["supercat_id"]
        new_dict_row["yolo_cx"] = dict_row["yolo_bbox"][0]
        new_dict_row["yolo_cy"] = dict_row["yolo_bbox"][1]
        new_dict_row["yolo_w"] = dict_row["yolo_bbox"][2]
        new_dict_row["yolo_h"] = dict_row["yolo_bbox"][3]        
        outputFileName = dict_row["image"]
        if (outputFileName+".txt") in file_list:
            content = read_file_from_s3("skm-yolo", "train/labels/"+outputFileName+".txt").strip("\n")
            for sub_content in content.split("\n"):
                set_key(new_dict_row, "class", int(sub_content.split(",")[0]))
                set_key(new_dict_row, "yolo_cx", float(sub_content.split(",")[1]))
                set_key(new_dict_row, "yolo_cy", float(sub_content.split(",")[2]))
                set_key(new_dict_row, "yolo_w", float(sub_content.split(",")[3]))
                set_key(new_dict_row, "yolo_h", float(sub_content.split(",")[4]))
            df = pd.concat([df, pd.DataFrame(new_dict_row)], ignore_index=True)
            write_file_to_s3("skm-yolo", df, outputFileName, "dataframe", "train/labels")
            
        else:
            df = pd.concat([df, pd.DataFrame([new_dict_row])], ignore_index=True)
            write_file_to_s3("skm-yolo", df, outputFileName, "dataframe", "train/labels")
            
        if len(list_all_objects_in_s3("skm-yolo", "train/labels")) % 20 == 0:
            print(len(list_all_objects_in_s3("skm-yolo", "train/labels")))

Writing to S3 completed for  train/labels/MTR_104_54
Writing to S3 completed for  train/labels/MTR_104_55
Writing to S3 completed for  train/labels/MTR_104_56
Writing to S3 completed for  train/labels/MTR_104_57
Writing to S3 completed for  train/labels/MTR_104_58
Writing to S3 completed for  train/labels/MTR_104_59
Writing to S3 completed for  train/labels/MTR_104_60
Writing to S3 completed for  train/labels/MTR_104_61
Writing to S3 completed for  train/labels/MTR_104_62
Writing to S3 completed for  train/labels/MTR_104_63
Writing to S3 completed for  train/labels/MTR_104_55
Writing to S3 completed for  train/labels/MTR_104_56
Writing to S3 completed for  train/labels/MTR_104_57
Writing to S3 completed for  train/labels/MTR_104_58
Writing to S3 completed for  train/labels/MTR_104_59
Writing to S3 completed for  train/labels/MTR_104_60
Writing to S3 completed for  train/labels/MTR_104_61
Writing to S3 completed for  train/labels/MTR_104_62
Writing to S3 completed for  train/labels/MTR_

In [29]:
print("done")

done


In [30]:
len(list_all_objects_in_s3("skm-yolo", "train/labels/"))


4469